In [1]:
import json
import os
from dotenv import load_dotenv
from langchain_litellm import ChatLiteLLMRouter
from litellm import Router


load_dotenv()

# 从环境变量加载 LLM 模型配置
model_list = json.loads(os.getenv("LLM_MODELS", "[]"))

litellm_router = Router(model_list=model_list)
llm = ChatLiteLLMRouter(router=litellm_router, model_name="glm5", temperature=0.1)

llm.invoke("hi")

c:\Apps\miniconda3\envs\learn\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `Generation` - serialized value may not be as expected [field_name='generations', input_value=ChatGeneration(text="Hi t...ls': {'reasoning': 0}})), input_type=ChatGeneration])
  PydanticSerializationUnexpectedValue(Expected `BaseMessage` - serialized value may not be as expected [field_name='message', input_value=AIMessage(content="Hi the...ils': {'reasoning': 0}}), input_type=AIMessage])
  PydanticSerializationUnexpectedValue(Expected `GenerationChunk` - serialized value may not be as expected [field_name='generations', input_value=ChatGeneration(text="Hi t...ls': {'reasoning': 0}})), input_type=ChatGeneration])
  PydanticSerializationUnexpectedValue(Expected `ChatGenerationChunk` - serialized value may not be as expected [field_name='generations', input_value=ChatGeneration(text="Hi t...ls': {'reasoning': 0}})), input_type=Ch

AIMessage(content="Hi there! I'm the GLM language model trained by Z.ai. How can I assist you today? Whether you have questions, need information, or just want to chat, I'm here to help. What would you like to discuss?", additional_kwargs={'provider_specific_fields': {'refusal': None}}, response_metadata={'token_usage': Usage(completion_tokens=50, prompt_tokens=6, total_tokens=56, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=0, rejected_prediction_tokens=None, text_tokens=None, image_tokens=None, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None)), 'model_group': 'glm5', 'model_group_alias': None, 'model_group_size': 1, 'attempted_retries': 0, 'max_retries': 2, 'deployment': 'zai/glm-5', 'model_info': {'id': '1219302a329bdd35c28ad7b6bb7f782201e0928aa21c3a391251a8a0749870e8', 'db_model': False}, 'ap

In [2]:
from pathlib import Path

folder_path = Path("../example_data/Journey_to_the_West")

all_text = ""

for txt_file in sorted(folder_path.glob("*.txt")):
    content = txt_file.read_text(encoding="utf-8")
    all_text += "\n\n" + content

len(all_text)

7749

In [3]:
from langchain_core.runnables import RunnableLambda
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import Dict, List


def split_into_chunks(text: str) -> List[Dict[str, str]]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=10000,
        chunk_overlap=1000,
        separators=[
            "\n\n",  # 段落分隔符
            "\n",  # 单个换行
            " ",  # 空格（英文单词分隔）
            ".",  # ASCII 句号 (U+002E)
            ",",  # ASCII 逗号 (U+002C)
            "\u200b",  # 零宽空格 常用于泰语/缅甸语/高棉语/日语，隐形分词
            "\uff0c",  # 全角逗号 “，” 中文常用逗号
            "\u3001",  # 顿号 “、” 中文/日文列举分隔符
            "\uff0e",  # 全角句号 “．” 常见于全角文本
            "\u3002",  # 句号 “。” 中文/日文句号
            "",  # 最后兜底：按单个字符切分
        ],
    )
    return [{"chunk": chunk} for chunk in splitter.split_text(text)]


text_chunks_chain = RunnableLambda(split_into_chunks)

In [4]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel
from langchain_core.output_parsers import StrOutputParser


summarize_chunk_prompt_template = """
Write a concise summary of the following text, and include the main details.
Text: {chunk}
"""

summarize_chunk_prompt = PromptTemplate.from_template(summarize_chunk_prompt_template)
summarize_chunk_chain = summarize_chunk_prompt | llm

summarize_map_chain = RunnableParallel(
    {"summary": summarize_chunk_chain | StrOutputParser()}
)

In [5]:
summarize_summaries_prompt_template = """
Write a concise summary of the following text, 
which joins several summaries, and include the main details.
Text: {summaries}
"""

summarize_summaries_prompt = PromptTemplate.from_template(
    summarize_summaries_prompt_template
)

summarize_reduce_chain = (
    RunnableLambda(lambda x: {"summaries": "\n".join([i["summary"] for i in x])})
    | summarize_summaries_prompt
    | llm
    | StrOutputParser()
)

In [6]:
map_reduce_chain = (
    text_chunks_chain | summarize_map_chain.map() | summarize_reduce_chain
)


summary = map_reduce_chain.invoke(all_text)

c:\Apps\miniconda3\envs\learn\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `Generation` - serialized value may not be as expected [field_name='generations', input_value=ChatGeneration(text='这...ls': {'reasoning': 0}})), input_type=ChatGeneration])
  PydanticSerializationUnexpectedValue(Expected `BaseMessage` - serialized value may not be as expected [field_name='message', input_value=AIMessage(content='这是...ils': {'reasoning': 0}}), input_type=AIMessage])
  PydanticSerializationUnexpectedValue(Expected `GenerationChunk` - serialized value may not be as expected [field_name='generations', input_value=ChatGeneration(text='这...ls': {'reasoning': 0}})), input_type=ChatGeneration])
  PydanticSerializationUnexpectedValue(Expected `ChatGenerationChunk` - serialized value may not be as expected [field_name='generations', input_value=ChatGeneration(text='这...ls': {'reasoning': 0}})), input_type=ChatGeneration]

In [7]:
print(summary)

本文讲述了孙悟空出世、学艺及大闹天宫的神话故事。仙石孕育的石猴因探得水帘洞被拜为“美猴王”，后拜菩提祖师为师获名“孙悟空”，习得长生术与七十二变。悟空在龙宫强取金箍棒，大闹地府勾销生死簿。玉帝招安其做弼马温，悟空嫌官小反下天庭自封“齐天大圣”，后因偷吃蟠桃、金丹惹祸。虽经十万天兵捉拿未果，最终被二郎神擒获，又在炼成火眼金睛后大闹天宫，终被如来佛祖压在五行山下，等待取经人。
